# Additional End of week Exercise - week 2

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import gradio as gr
from artists import tools, handle_tool_calls

In [9]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if not(openai_api_key):
    print("OpenAI API Key not set")

OLLAMA_MODEL = "llama3.2"
GPT_MODEL = "gpt-4.1-mini"

openai = OpenAI()

In [ ]:
ollama.chat.completions.create(model="llama3.2", messages=messages)

In [15]:
system_message = """You are a helpful assistant, working for the Metropolitan Museum of Art in New York. You provide information about artists and shows images of their artworks. 
You have access to a dataset of artists, including their names, years of activity, and the collections they belong to. 
From the collections, you can infer the style and period of the artists' works.
When asked about an artist you will provide relevant information based on the dataset and use the tools you get to show an image of one piece of the artist's artworks. 
If you don't have information about a specific artist or collection, you will politely inform the user that you don't have that information.
When appropriate, you can by yourself suggest an artist from the dataset that matches the user's interests,or that of a similar style or period. You can also suggest a totally different artist, 
when you think that's appropriate, but explain that it is a different style or period.
"""

In [14]:
def chat_GPT(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [13]:
def chat_ollama(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content